# SHAP Prism quick start

This notebook uses the fully synthetic delivery scenario shown in the graphical abstract. All 342 rows are generated reproducibly in memory; no data file is downloaded.

In [ ]:
!pip install shap-prism

## Graphical-abstract example

The example has the same five features, delivery modes, route-distance bands, group sizes, and displayed group means as the graphical abstract. The feature table and the illustrative SHAP matrix stay aligned row for row.

In [ ]:
# @title Create the graphical-abstract data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(4701)
bands = ["0–2 km", "2–5 km", "5–10 km", "10–20 km"]
counts = [120, 96, 72, 54]
bounds = [(0, 2), (2, 5), (5, 10), (10, 20)]
modes = ["Cargo bike", "Car", "Van"]

route_band = np.concatenate([
    np.repeat(band, count) for band, count in zip(bands, counts, strict=True)
])
route_distance = np.concatenate([
    rng.uniform(low, high, count)
    for (low, high), count in zip(bounds, counts, strict=True)
])
delivery_mode = np.concatenate([
    rng.permutation(np.tile(modes, count // 3)) for count in counts
])
traffic_index = rng.uniform(0, 1, len(route_band))
heavy_rain = np.repeat(["No", "Yes"], [171, 171])
drop_off = np.repeat(["Parcel locker", "Reception", "Doorstep"], [117, 114, 111])
rng.shuffle(heavy_rain)
rng.shuffle(drop_off)

features = pd.DataFrame({
    "Route distance": route_distance,
    "Delivery mode": pd.Categorical(delivery_mode, modes, ordered=True),
    "Traffic index": traffic_index,
    "Heavy rain": pd.Categorical(heavy_rain, ["No", "Yes"], ordered=True),
    "Drop-off type": pd.Categorical(
        drop_off, ["Parcel locker", "Reception", "Doorstep"], ordered=True
    ),
})
groups = pd.Series(
    pd.Categorical(route_band, bands, ordered=True), name="Route-distance band"
)

# Delivery-mode effects reproduce the four group means in the artwork.
mode_centres = {
    "0–2 km": [-1.35, -0.30, 0.75],
    "2–5 km": [-0.65, -0.20, 0.25],
    "5–10 km": [0.65, -0.20, -0.15],
    "10–20 km": [1.55, -0.20, -0.60],
}
mode_index = {mode: index for index, mode in enumerate(modes)}
delivery_shap = np.array([
    mode_centres[band][mode_index[mode]]
    for band, mode in zip(route_band, delivery_mode, strict=True)
])
jitter = rng.normal(0, 0.13, len(route_band))
for band in bands:
    in_band = route_band == band
    jitter[in_band] -= jitter[in_band].mean()
delivery_shap += jitter

shap_values = pd.DataFrame({
    "Route distance": 0.20 * (route_distance - 4) + rng.normal(0, 0.08, len(route_band)),
    "Delivery mode": delivery_shap,
    "Traffic index": 1.15 * (traffic_index - 0.5) + rng.normal(0, 0.05, len(route_band)),
    "Heavy rain": np.where(heavy_rain == "Yes", 0.28, -0.12) + rng.normal(0, 0.02, len(route_band)),
    "Drop-off type": pd.Series(drop_off).map(
        {"Parcel locker": -0.20, "Reception": 0.0, "Doorstep": 0.17}
    ).to_numpy() + rng.normal(0, 0.02, len(route_band)),
})

## Global summary

In [ ]:
%config InlineBackend.figure_format = "retina"
from shap_prism import plot_summary

summary = plot_summary(
    shap_values, features, category_key_placement="bottom",
)
plt.show()

## SHAP Prism view

In [ ]:
%config InlineBackend.figure_format = "retina"
from shap_prism import plot_prism

prism = plot_prism(
    shap_values,
    features,
    groups,
    "Delivery mode",
    focal_kind="categorical",
    group_order=bands,
    category_order=modes,
)
plt.show()

For your own analysis, replace the synthetic `features` and `shap_values` with an aligned feature table and SHAP matrix from your explainer.